# CrimeScope ML — 02 (UK). Ingest & Geographies (Lakehouse)

**Description:** Pull 60 months of `data.police.uk` street-level crime CSVs for
every England & Wales territorial police force, stage them in a UC Volume, land
them as a Delta table, and aggregate to monthly counts per LSOA and MSOA with a
violent / property / other split. Also fetches ONS LSOA + MSOA 2021 boundaries
and the LSOA→MSOA lookup.

**Lakehouse features used:**
- Unity Catalog (`varanasi.default.*`)
- UC Volume `varanasi.default.ml_data_uk` for raw archive staging
- Delta Lake with `OPTIMIZE` + `ZORDER` on `lsoa_code` / `month_start`
- Data-quality assertions (LSOA join rate, monthly coverage per force)
- Table `COMMENT`s for governance

**Tables written:**
- `uk_crime_raw` — every street-level row, 60 months, all forces
- `uk_crime_monthly_lsoa` — LSOA × month counts (overall / violent / property)
- `uk_crime_monthly_msoa` — MSOA × month counts (rolled up via lookup)
- `uk_lsoa_boundaries` — ONS LSOA 2021 BGC polygons (WKT)
- `uk_msoa_boundaries` — ONS MSOA 2021 BSC polygons (WKT)
- `uk_lsoa_to_msoa_lookup` — official ONS lookup

In [ ]:
spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")
display(spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema"))

---
## 1. Create the UK staging volume

In [ ]:
spark.sql("""
  CREATE VOLUME IF NOT EXISTS varanasi.default.ml_data_uk
  COMMENT 'Raw data staging for CrimeScope UK & Wales ML pipeline'
""")
print("Volume varanasi.default.ml_data_uk ready")

---
## 2. Pull data.police.uk monthly archives (60 months)

Each archive is a single zip per month containing `<month>/<month>-<force>-street.csv`
for every territorial force. We stream the zip into the UC Volume so we don't keep
~10 GB of compressed data in driver memory. The download is idempotent — already-staged
months are skipped.

In [ ]:
import io
import os
import urllib.request
import urllib.error
from datetime import date, timedelta
from pathlib import Path

VOLUME_RAW = "/Volumes/varanasi/default/ml_data_uk/raw/police_uk"
ARCHIVE_BASE = "https://data.police.uk/data/archive"
N_MONTHS = 60

os.makedirs(VOLUME_RAW, exist_ok=True)


def months_back(n: int) -> list[str]:
    today = date.today().replace(day=1)
    # data.police.uk publishes ~6 weeks after month-end, so we lag by 2 months
    cursor = (today - timedelta(days=62)).replace(day=1)
    out = []
    for _ in range(n):
        out.append(cursor.strftime("%Y-%m"))
        cursor = (cursor.replace(day=1) - timedelta(days=1)).replace(day=1)
    return list(reversed(out))


TARGET_MONTHS = months_back(N_MONTHS)
print(f"Target months: {TARGET_MONTHS[0]} .. {TARGET_MONTHS[-1]} ({len(TARGET_MONTHS)} months)")

downloaded = 0
skipped = 0
for ym in TARGET_MONTHS:
    dst = f"{VOLUME_RAW}/{ym}.zip"
    if os.path.exists(dst) and os.path.getsize(dst) > 1_000_000:
        skipped += 1
        continue
    url = f"{ARCHIVE_BASE}/{ym}.zip"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "CrimeScope-UK/1.0"})
        with urllib.request.urlopen(req, timeout=300) as resp, open(dst, "wb") as f:
            while True:
                buf = resp.read(1 << 20)
                if not buf:
                    break
                f.write(buf)
        downloaded += 1
        print(f"  fetched {ym}.zip ({os.path.getsize(dst) / 1_048_576:.1f} MB)")
    except urllib.error.HTTPError as e:
        # data.police.uk publishes monthly archives one per month; missing months are normal at the edge
        print(f"  skip {ym} ({e})")
    except Exception as e:  # noqa: BLE001
        print(f"  error {ym}: {e}")

print(f"\nDownloaded {downloaded} new, skipped {skipped} existing.")

---
## 3. Extract archives in parallel into a flat CSV staging dir

Each `YYYY-MM.zip` archive contains ~43 force CSVs (one per UK police force)
plus outcomes/stop-and-search files. We extract just the `*-street.csv`
files into a flat directory on the same UC Volume. Extraction is local-disk
I/O bound and parallelizes well across a thread pool on the driver.

After this cell, the staging dir contains ~2,500 CSVs (~60 months × ~43 forces),
ready for a single distributed `spark.read.csv` in the next step.

In [ ]:
import shutil
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed

STAGE_DIR = Path(VOLUME_RAW) / "_extracted_street"
if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True, exist_ok=True)

def _extract_one(arc_path: Path) -> int:
    n = 0
    month_tag = arc_path.stem  # e.g. 2021-03
    with zipfile.ZipFile(arc_path) as zf:
        for member in zf.namelist():
            if not member.endswith("-street.csv"):
                continue
            # member looks like '2021-03/2021-03-avon-and-somerset-street.csv'
            out_name = f"{month_tag}__{Path(member).name}"
            out_path = STAGE_DIR / out_name
            with zf.open(member) as src, open(out_path, "wb") as dst:
                shutil.copyfileobj(src, dst, length=1 << 20)
            n += 1
    return n

archives = sorted(Path(VOLUME_RAW).glob("*.zip"))
print(f"Extracting *-street.csv from {len(archives)} archives...")

total_csvs = 0
with ThreadPoolExecutor(max_workers=16) as pool:
    futs = {pool.submit(_extract_one, a): a for a in archives}
    for fut in as_completed(futs):
        arc = futs[fut]
        try:
            n = fut.result()
            total_csvs += n
        except Exception as e:  # noqa: BLE001
            print(f"  ! {arc.name}: {e}")

print(f"Extracted {total_csvs} CSV files into {STAGE_DIR}")

---
## 4. Bulk-load all street CSVs into `uk_crime_raw` with one Spark read

Single distributed `spark.read.csv` over the staging directory. This is
~100x faster than the per-archive append loop because Spark parallelizes
the read across executors and writes a single large Delta commit instead
of 2,500 small ones.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
)

VIOLENT_CATS = [
    "Violence and sexual offences",
    "Robbery",
    "Possession of weapons",
    "Public order",
]
PROPERTY_CATS = [
    "Burglary",
    "Vehicle crime",
    "Theft from the person",
    "Bicycle theft",
    "Other theft",
    "Shoplifting",
    "Criminal damage and arson",
]

# Police.uk CSV header is consistent across all months.
CSV_SCHEMA = StructType([
    StructField("Crime ID",                StringType()),
    StructField("Month",                   StringType()),
    StructField("Reported by",             StringType()),
    StructField("Falls within",            StringType()),
    StructField("Longitude",               DoubleType()),
    StructField("Latitude",                DoubleType()),
    StructField("Location",                StringType()),
    StructField("LSOA code",               StringType()),
    StructField("LSOA name",               StringType()),
    StructField("Crime type",              StringType()),
    StructField("Last outcome category",   StringType()),
    StructField("Context",                 StringType()),
])

STAGE_DIR = f"{VOLUME_RAW}/_extracted_street"
TABLE = "varanasi.default.uk_crime_raw"

raw = (spark.read
            .option("header", True)
            .option("multiLine", False)
            .option("escape", '"')
            .schema(CSV_SCHEMA)
            .csv(f"{STAGE_DIR}/*.csv"))

# Restrict to England & Wales LSOAs and shape to canonical schema.
cleaned = (raw
    .withColumnRenamed("Crime ID",   "crime_id")
    .withColumnRenamed("Month",      "month")
    .withColumnRenamed("Falls within", "force")
    .withColumnRenamed("Longitude",  "longitude")
    .withColumnRenamed("Latitude",   "latitude")
    .withColumnRenamed("LSOA code",  "lsoa_code")
    .withColumnRenamed("LSOA name",  "lsoa_name")
    .withColumnRenamed("Crime type", "crime_type")
    .withColumnRenamed("Last outcome category", "last_outcome")
    .drop("Reported by", "Location", "Context")
    .filter(F.col("lsoa_code").isNotNull())
    .filter(F.col("lsoa_code").rlike("^(E0|W0)"))
    .withColumn("month_start", F.to_date(F.concat_ws("-", F.col("month"), F.lit("01"))))
    .withColumn("category", F.coalesce(F.col("crime_type"), F.lit("Other crime")))
    .withColumn("is_violent",  F.col("category").isin(VIOLENT_CATS).cast("tinyint"))
    .withColumn("is_property", F.col("category").isin(PROPERTY_CATS).cast("tinyint"))
    .select(
        "crime_id", "month", "force",
        "longitude", "latitude",
        "lsoa_code", "lsoa_name",
        "crime_type", "last_outcome",
        "month_start", "category",
        "is_violent", "is_property",
    ))

(cleaned.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE))

spark.sql(f"""
  ALTER TABLE {TABLE}
  SET TBLPROPERTIES (
    'comment' = 'Raw street-level crime rows from data.police.uk for England & Wales (60 months).'
  )
""")
spark.sql(f"OPTIMIZE {TABLE} ZORDER BY (lsoa_code, month_start)")
n_rows = spark.table(TABLE).count()
print(f"uk_crime_raw written + optimized: {n_rows:,} rows")

# Free up the staging directory so we don't retain ~50 GB of CSVs.
import shutil
shutil.rmtree(STAGE_DIR, ignore_errors=True)
print("Cleaned up staging dir")

---
## 5. ONS boundary downloads (LSOA + MSOA 2021)

In [ ]:
import json
import urllib.parse

ONS_LSOA = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3/"
    "FeatureServer/0/query"
)
ONS_MSOA = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BSC_V3/"
    "FeatureServer/0/query"
)
PAGE = 2000


def fetch_layer(base: str, code_field: str, name_field: str) -> list[dict]:
    out = []
    offset = 0
    while True:
        params = {
            "where": "1=1",
            "outFields": f"{code_field},{name_field},LAT,LONG",
            "outSR": "4326",
            "f": "geojson",
            "resultOffset": str(offset),
            "resultRecordCount": str(PAGE),
            "orderByFields": code_field,
        }
        url = f"{base}?{urllib.parse.urlencode(params)}"
        req = urllib.request.Request(url, headers={"User-Agent": "CrimeScope-UK/1.0"})
        with urllib.request.urlopen(req, timeout=180) as resp:
            page = json.loads(resp.read())
        feats = page.get("features", [])
        if not feats:
            break
        out.extend(feats)
        if len(feats) < PAGE:
            break
        offset += PAGE
        print(f"    offset={offset:>5}  total={len(out):>6}")
    return out


def features_to_pdf(feats: list[dict], code_field: str, name_field: str, label: str) -> pd.DataFrame:
    def ring_wkt(r):
        return ", ".join(f"{x} {y}" for x, y in r)

    def poly_wkt(p):
        return "(" + ", ".join(f"({ring_wkt(r)})" for r in p) + ")"

    rows = []
    for f in feats:
        p = f.get("properties") or {}
        code = p.get(code_field)
        name = p.get(name_field) or code
        if not code:
            continue
        geom = f.get("geometry") or {}
        if geom.get("type") == "Polygon":
            wkt = "POLYGON " + poly_wkt(geom["coordinates"])
        elif geom.get("type") == "MultiPolygon":
            wkt = "MULTIPOLYGON (" + ", ".join(poly_wkt(p_) for p_ in geom["coordinates"]) + ")"
        else:
            continue
        rows.append({
            "tract_geoid": code,
            "NAMELSAD": name,
            "wkt": wkt,
            "ALAND": None,
            "lat": p.get("LAT"),
            "lng": p.get("LONG"),
        })
    print(f"  {label}: {len(rows)} polygons")
    return pd.DataFrame(rows)


print("[boundaries] LSOA…")
lsoa_feats = fetch_layer(ONS_LSOA, "LSOA21CD", "LSOA21NM")
lsoa_pdf = features_to_pdf(lsoa_feats, "LSOA21CD", "LSOA21NM", "LSOA")

print("[boundaries] MSOA…")
msoa_feats = fetch_layer(ONS_MSOA, "MSOA21CD", "MSOA21NM")
msoa_pdf = features_to_pdf(msoa_feats, "MSOA21CD", "MSOA21NM", "MSOA")

spark.createDataFrame(lsoa_pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("varanasi.default.uk_lsoa_boundaries")
spark.createDataFrame(msoa_pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("varanasi.default.uk_msoa_boundaries")
spark.sql("ALTER TABLE varanasi.default.uk_lsoa_boundaries SET TBLPROPERTIES ('comment' = 'ONS LSOA Dec 2021 BGC boundaries (England & Wales).')")
spark.sql("ALTER TABLE varanasi.default.uk_msoa_boundaries SET TBLPROPERTIES ('comment' = 'ONS MSOA Dec 2021 BSC boundaries (England & Wales).')")
print("Boundaries written.")

---
## 6. LSOA → MSOA lookup

In [ ]:
# Use the ONS Postcode Directory's official LSOA(2021) → MSOA(2021) → LAD lookup
LOOKUP_URL = (
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
    "Output_Areas_to_Lower_Layer_Super_Output_Areas_to_Middle_Layer_Super_Output_Areas_to_Local_Authority_Districts_December_2021_Lookup_EW/"
    "FeatureServer/0/query"
)

all_rows = []
offset = 0
while True:
    params = {
        "where": "1=1",
        "outFields": "LSOA21CD,MSOA21CD,LAD22CD,LAD22NM",
        "returnGeometry": "false",
        "f": "json",
        "resultOffset": str(offset),
        "resultRecordCount": "2000",
        "orderByFields": "LSOA21CD",
    }
    url = f"{LOOKUP_URL}?{urllib.parse.urlencode(params)}"
    req = urllib.request.Request(url, headers={"User-Agent": "CrimeScope-UK/1.0"})
    with urllib.request.urlopen(req, timeout=180) as resp:
        page = json.loads(resp.read())
    feats = page.get("features", [])
    if not feats:
        break
    all_rows.extend([f["attributes"] for f in feats])
    if len(feats) < 2000:
        break
    offset += 2000
    print(f"  lookup offset={offset:>5}  total={len(all_rows):>6}")

lookup_pdf = pd.DataFrame(all_rows).rename(columns={
    "LSOA21CD": "lsoa_code", "MSOA21CD": "msoa_code",
    "LAD22CD": "lad_code", "LAD22NM": "lad_name",
}).drop_duplicates(subset=["lsoa_code"])
print(f"Lookup rows: {len(lookup_pdf):,}")

spark.createDataFrame(lookup_pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("varanasi.default.uk_lsoa_to_msoa_lookup")
spark.sql("ALTER TABLE varanasi.default.uk_lsoa_to_msoa_lookup SET TBLPROPERTIES ('comment' = 'ONS LSOA(2021) -> MSOA(2021) -> LAD(2022) lookup for England & Wales.')")

---
## 7. Aggregate to monthly counts (LSOA + MSOA)

In [ ]:
spark.sql("""
  CREATE OR REPLACE TABLE varanasi.default.uk_crime_monthly_lsoa
  COMMENT 'Per-LSOA monthly crime counts with violent / property split (data.police.uk).'
  AS
  SELECT
    lsoa_code,
    ANY_VALUE(lsoa_name) AS lsoa_name,
    month_start,
    COUNT(*) AS incident_count,
    CAST(SUM(is_violent) AS INT) AS violent_count,
    CAST(SUM(is_property) AS INT) AS property_count
  FROM varanasi.default.uk_crime_raw
  WHERE lsoa_code IS NOT NULL
  GROUP BY lsoa_code, month_start
""")
spark.sql("OPTIMIZE varanasi.default.uk_crime_monthly_lsoa ZORDER BY (lsoa_code, month_start)")
display(spark.sql("SELECT * FROM varanasi.default.uk_crime_monthly_lsoa LIMIT 10"))

In [ ]:
spark.sql("""
  CREATE OR REPLACE TABLE varanasi.default.uk_crime_monthly_msoa
  COMMENT 'Per-MSOA monthly crime counts (rolled up via ONS LSOA->MSOA lookup).'
  AS
  SELECT
    l.msoa_code,
    c.month_start,
    SUM(c.incident_count) AS incident_count,
    SUM(c.violent_count) AS violent_count,
    SUM(c.property_count) AS property_count
  FROM varanasi.default.uk_crime_monthly_lsoa c
  JOIN varanasi.default.uk_lsoa_to_msoa_lookup l
    ON c.lsoa_code = l.lsoa_code
  GROUP BY l.msoa_code, c.month_start
""")
spark.sql("OPTIMIZE varanasi.default.uk_crime_monthly_msoa ZORDER BY (msoa_code, month_start)")

---
## 7. Data-quality assertions

In [ ]:
n_raw = spark.table("varanasi.default.uk_crime_raw").count()
n_lookup = spark.table("varanasi.default.uk_lsoa_to_msoa_lookup").count()
joined = spark.sql("""
  SELECT COUNT(*) AS n,
         SUM(CASE WHEN l.msoa_code IS NULL THEN 0 ELSE 1 END) AS matched
  FROM varanasi.default.uk_crime_raw r
  LEFT JOIN varanasi.default.uk_lsoa_to_msoa_lookup l
    ON r.lsoa_code = l.lsoa_code
  WHERE r.lsoa_code IS NOT NULL
""").first()

match_rate = joined.matched / max(joined.n, 1)
print(f"Raw crime rows:          {n_raw:,}")
print(f"Lookup rows:             {n_lookup:,}")
print(f"LSOA join rate:          {match_rate * 100:.2f}%")

assert n_raw > 5_000_000, f"Crime row count looks too low ({n_raw:,})"
assert match_rate > 0.95, f"LSOA join rate too low ({match_rate:.2%})"

coverage = spark.sql("""
  SELECT force, COUNT(DISTINCT month_start) AS n_months
  FROM varanasi.default.uk_crime_raw
  GROUP BY force
""").toPandas()
too_thin = coverage[coverage["n_months"] < 24]
print(f"\nForces with <24 months of data: {len(too_thin)}")
if len(too_thin) > 0:
    print(too_thin)
# Hard assertion: at least 30 forces should have >=40 months
assert (coverage["n_months"] >= 40).sum() >= 30, "Coverage check failed"
print("\nAll DQ assertions passed.")